# VLM router benchmark (Colab + vLLM)

Measures the router call from the plan: **MetaDrive frame + short state text → one letter (A/B/C) → skill**.

- A = `CRUISE_RIGHT`, B = `OVERTAKE_LEFT`, C = `RETURN_RIGHT`
- `frames.zip` has 30 labelled MetaDrive top-down frames (6 each: clear, overtake, blocked, return, keep_left).

**Runtime:** Runtime → Change runtime type → GPU. **L4 or A100 recommended.** A T4 (free tier) should run the 3B in fp16;
for 7B on a T4 use the `-AWQ` model id. Run the cells top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install vllm

## 1. Upload `frames.zip`

In [ ]:
from google.colab import files
files.upload()          # choose frames.zip from the vlm-colab folder
!unzip -oq frames.zip -d /content && ls /content/frames | head

## 2. Load the model with vLLM

- `enable_prefix_caching`: the fixed system prompt's KV cache is reused across calls.
- `min_pixels/max_pixels`: bounds on image tokens (Qwen2.5-VL: 1 token per 28×28 px). Small images stay small.
- `allowed_token_ids`: the model can only output A, B or C, so decoding is exactly 1 step, and the logprobs give a confidence.

In [ ]:
import base64, io, json, random, time, statistics as st
import torch, pandas as pd
from PIL import Image
from vllm import LLM, SamplingParams

MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"   # also try "Qwen/Qwen2.5-VL-7B-Instruct" (L4/A100) or "Qwen/Qwen2.5-VL-7B-Instruct-AWQ" (T4)
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "half"

t = time.time()
llm = LLM(model=MODEL, dtype=DTYPE, max_model_len=4096, limit_mm_per_prompt={"image": 1},
          mm_processor_kwargs={"min_pixels": 28 * 28, "max_pixels": 1600 * 28 * 28},
          enable_prefix_caching=True, gpu_memory_utilization=0.85, seed=0)
print(f"model load: {time.time() - t:.1f} s")

tok = llm.get_tokenizer()
LETTERS = ["A", "B", "C"]
LETTER_IDS = [tok.encode(c, add_special_tokens=False)[0] for c in LETTERS]
ONE_LETTER = SamplingParams(temperature=0, max_tokens=1, allowed_token_ids=LETTER_IDS, logprobs=3)
FREE_TEXT = SamplingParams(temperature=0, max_tokens=80)

## 3. Prompt and call helper

In [ ]:
SYSTEM = """You are the skill router for a self-driving car. You see a bird's-eye view of the road.
The image shows our side of a highway: the yellow line on the left is the road centre, then the LEFT lane,
a dashed white divider, then the RIGHT lane, then the road edge. Our car is GREEN, near the bottom,
driving UP the image. Every other car is parked (speed 0).

Choose the skill our car should run now:
A = CRUISE_RIGHT: stay in / go to the right lane and keep driving. Use this when the right lane ahead is clear,
    OR when a car blocks the right lane ahead but the left lane next to us is occupied (we must wait).
B = OVERTAKE_LEFT: move to / stay in the left lane. Use this when a car blocks the right lane ahead (or is still
    ahead of us on the right) and the left lane is free.
C = RETURN_RIGHT: go back to the right lane. Use this when we are in the left lane and the car we passed is now
    behind us, so the right lane ahead is clear.

Reply with exactly one letter: A, B or C."""

def data_uri(img):
    b = io.BytesIO(); img.save(b, format="PNG")
    return "data:image/png;base64," + base64.b64encode(b.getvalue()).decode()

def ask(img, state, sp=ONE_LETTER, system=SYSTEM):
    """One router call, batch size 1. Order = fixed system prompt -> image -> short state text (cache-friendly)."""
    content = ([{"type": "image_url", "image_url": {"url": data_uri(img)}}] if img is not None else [])
    content.append({"type": "text", "text": state})
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": content}]
    t = time.perf_counter()
    out = llm.chat(msgs, sp, use_tqdm=False)[0]
    ms = (time.perf_counter() - t) * 1000
    o = out.outputs[0]
    conf = None
    if o.logprobs:
        lp = {v.decoded_token: v.logprob for v in o.logprobs[0].values()}
        z = {k: torch.tensor(lp[k]).exp().item() for k in LETTERS if k in lp}
        s = sum(z.values()) or 1.0
        conf = round(z.get(o.text, 0.0) / s, 3)
    return dict(text=o.text.strip(), ms=ms, prompt_tok=len(out.prompt_token_ids), out_tok=len(o.token_ids), conf=conf)

def fresh(img):
    """Change one pixel so the image misses the cache, like a new camera frame."""
    img = img.copy()
    img.putpixel((random.randrange(img.width), random.randrange(img.height)), (random.randrange(256),) * 3)
    return img

labels = json.load(open("/content/frames/labels.json"))
load = lambda name: Image.open(f"/content/frames/{name}").convert("RGB")
state = lambda lab: f"Current skill: {lab['current_skill']}. Ego speed: 20 m/s. Trigger: shield override."

for _ in range(3):                      # warm-up (CUDA graphs, kernels, prefix cache of the system prompt)
    ask(fresh(load(labels[0]["file"])), state(labels[0]))

## 4. Accuracy + latency on all 30 frames, at several image sizes

Bigger images = more image tokens = slower, but small cars may only be visible at higher resolution.
A model that always answers A scores 40%.

In [ ]:
SCALES = [1.0, 2.0, 3.0]                # 120x780 px native -> x2 -> x3
rows = []
for s in SCALES:
    for lab in labels:
        im = load(lab["file"])
        im = im.resize((int(im.width * s), int(im.height * s)))
        r = ask(fresh(im), state(lab))
        rows.append(dict(scale=s, size=f"{im.width}x{im.height}", kind=lab["kind"], want=lab["label"], **r,
                         ok=r["text"][:1] == lab["label"]))
df = pd.DataFrame(rows)
summary = df.groupby(["scale", "size"]).agg(accuracy=("ok", "mean"), median_ms=("ms", "median"),
                                             p90_ms=("ms", lambda x: x.quantile(0.9)), prompt_tok=("prompt_tok", "median"))
display(summary.round(3))
display(df.pivot_table(index="kind", columns="scale", values="ok", aggfunc="mean").round(2))

## 5. Where the time goes

- **text-only**: same prompt with no image -> LLM prefill cost without the vision encoder.
- **fresh image** at several sizes -> vision encoder + prefill as image tokens grow.
- **identical repeat**: same image + text again -> prefix-cache hit (the floor for one call).
- **free text**: 80 output tokens of reasoning instead of 1 letter -> decode cost.

In [ ]:
base = load(labels[6]["file"])
def med(fn, n=5):
    rs = [fn() for _ in range(n)]
    return dict(median_ms=round(st.median(r["ms"] for r in rs), 1), prompt_tok=rs[0]["prompt_tok"], out_tok=rs[0]["out_tok"])

timing = {}
timing["text only (no image)"] = med(lambda: ask(None, f"{random.random()} Current skill: CRUISE_RIGHT."))
for w, h in [(120, 780), (240, 1560), (448, 252), (640, 360), (896, 504), (1280, 720)]:
    im = base.resize((w, h))
    timing[f"fresh image {w}x{h}"] = med(lambda: ask(fresh(im), "Current skill: CRUISE_RIGHT. Ego speed: 20 m/s."))
same = base.copy()
timing["identical repeat (cache hit)"] = med(lambda: ask(same, "Current skill: CRUISE_RIGHT. Ego speed: 20 m/s."))
free_sys = SYSTEM.replace("Reply with exactly one letter: A, B or C.", "Explain your reasoning in one or two sentences, then give the letter.")
timing["free text, 80 tokens max"] = med(lambda: ask(fresh(base), "Current skill: CRUISE_RIGHT.", sp=FREE_TEXT, system=free_sys), n=3)
display(pd.DataFrame(timing).T)
print("example free-text answer:", ask(fresh(base), "Current skill: CRUISE_RIGHT.", sp=FREE_TEXT, system=free_sys)["text"])

## 6. Save results

Download `results.json` and share it (or paste the tables). To compare models, change `MODEL` in step 2,
then **Runtime → Restart session** and run everything again (vLLM keeps the GPU memory until restart).

In [ ]:
json.dump(dict(model=MODEL, gpu=torch.cuda.get_device_name(0), frames=rows,
               summary=summary.reset_index().to_dict("records"), timing=timing),
          open("/content/results.json", "w"), indent=1, default=str)
files.download("/content/results.json")